In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [1]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [5]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your operating system:
   - macOS: download the `.pkg`
   - Windows: download the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

3. To test the local server, run:
   ```bash
   curl http://localhost:11434
   ```

If needed, install the Python client with:
```bash
pip install ollama
```


In [6]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I don’t see any FAQ entry about **Olama** specifically.

The closest relevant answer is that you **can run the course locally instead of Codespaces** if you’re comfortable setting up the needed tools such as **Python, `uv`, Jupyter, Docker, and any other tools needed for the module**. If you run locally, you should **document your setup and keep your environment reproducible**.

If you meant a specific local setup for **Ollama**, that isn’t covered in the provided FAQ context.


In [7]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Maybe — it depends on the course’s enrollment rules and whether it’s still open.\n\nIf you want, I can help you figure it out quickly. Check for:\n- **Enrollment deadline**\n- **Prerequisites or approval required**\n- **Whether there are still seats available**\n- **Late add / audit options**\n\nIf you’re asking what to say, you could message the instructor or registrar with:\n\n> Hi, I just discovered this course and I’m very interested. Is it still possible to join, or is late enrollment available?\n\nIf you want, I can also help you draft a more polished email.'

In [8]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [10]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [11]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late start"}', call_id='call_47rq5z6IH7qVwrBsJGfd1B12', name='search', type='function_call', id='fc_0083041b58d5534e006a35a8fddb808190be1a833d0cd9b699', namespace=None, status='completed')]

In [12]:
response.output[0]

ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late start"}', call_id='call_47rq5z6IH7qVwrBsJGfd1B12', name='search', type='function_call', id='fc_0083041b58d5534e006a35a8fddb808190be1a833d0cd9b699', namespace=None, status='completed')

In [13]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [15]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch

In [16]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [17]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late start"}', call_id='call_47rq5z6IH7qVwrBsJGfd1B12', name='search', type='function_call', id='fc_0083041b58d5534e006a35a8fddb808190be1a833d0cd9b699', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_47rq5z6IH7qVwrBsJGfd1B12',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer"

In [19]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

Yes — you can still join. You can start anytime, and the videos/materials are available.

If you want a certificate, though, you’ll need to submit your project while the course is still accepting submissions.


In [20]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(771, 47)

In [21]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(771, 47)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.00014385


## Agentic Loop

### Developer Prompt

In [22]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

### A function-call helper

We'll be running function calls repeatedly inside the loop, so let's wrap that in a small helper. It turns the JSON arguments into a Python dict, calls the right function, and serializes the result. We only have one tool for now, so we dispatch on the function name directly.


In [23]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

### Processing one response

In [35]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered the course can I join enrollment registration new student"}


In [36]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered the course can I join enrollment registration new student"}', call_id='call_maQJSw0fPoPxAClWIkEwCKTC', name='search', type='function_call', id='fc_04e52e5b62d6bef9006a365d55840481908abd710e95ca5611', namespace=None, status='completed')]

### The full agent loop

We wrap this in a while loop. The loop keeps calling the model until it returns a response without any function calls. We also keep an iteration counter so we can see how many round-trips happened.

In [38]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"self-paced mode certificate join course now can I still join project submissions open registration not needed"}
iteration #2...
ASSISTANT:
Yes, you can still join the course.

A couple of important notes:
- You can start learning and submitting homework even if you didn’t register ahead of time.
- If you want a certificate, you need to complete the capstone/project while submissions are still open, and the course needs to be running live for the peer-review part.

If you want, I can also tell you the best way to start the course from here. Are there other areas you want to explore?


### Wrapping it in a function

Let's wrap the loop in a function so we can reuse it. The function takes the instructions and the question as parameters, and returns the final answer.

In [39]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [ ]:
agent_loop(instructions, "How do I run Olama locally?") # Try it with a question that has a typo

iteration #1...
function_call: search {"query":"Olama locally run local install setup"}
function_call: search {"query":"Ollama local run install setup"}
function_call: search {"query":"local model server Ollama run locally"}
iteration #2...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: install the `.pkg`
     - **Windows**: install the `.msi`
     - **Linux**:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This will download the model, start it locally, and open a chat-like terminal.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response showing available models.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
   

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: install the `.pkg`\n     - **Windows**: install the `.msi`\n     - **Linux**:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model, start it locally, and open a chat-like terminal.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response showing available models.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, try restarting the server:\n```bash\nollama serve\n``

In [41]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discover course can I still join enrollment late registration"}
function_call: search {"query":"course FAQ join after start can I still enroll discovered the course"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still being accepted. You can also start learning and submitting homework while the form is open, and registration isn’t strictly required.

If you want, I can also help you figure out the best way to start catching up.


'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still being accepted. You can also start learning and submitting homework while the form is open, and registration isn’t strictly required.\n\nIf you want, I can also help you figure out the best way to start catching up.'

### Restricting off-topic questions

In [42]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit d4 d5 c4"}
function_call: search {"query":"queen gambit FAQ chess opening"}
function_call: search {"query":"gambit chess queen's gambit definition"}
iteration #2...
ASSISTANT:
A **queen’s gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the **c-pawn** as a gambit to tempt Black into taking it, in order to gain control of the center and develop pieces more actively.

A few quick notes:
- It’s one of the most famous and classical chess openings.
- Despite the name, the pawn is usually not a real “queen” pawn sacrifice in the long term.
- If Black accepts the gambit, it’s called the **Queen’s Gambit Accepted**.
- If Black declines it, it’s called the **Queen’s Gambit Declined**.

If you want, I can also explain **why it’s called a gambit** or show the **main variations**.


'A **queen’s gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** as a gambit to tempt Black into taking it, in order to gain control of the center and develop pieces more actively.\n\nA few quick notes:\n- It’s one of the most famous and classical chess openings.\n- Despite the name, the pawn is usually not a real “queen” pawn sacrifice in the long term.\n- If Black accepts the gambit, it’s called the **Queen’s Gambit Accepted**.\n- If Black declines it, it’s called the **Queen’s Gambit Declined**.\n\nIf you want, I can also explain **why it’s called a gambit** or show the **main variations**.'

In [43]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit opening chess FAQ"}
function_call: search {"query":"gambit chess queen FAQ"}
function_call: search {"query":"queen gambit course FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry for “queen gambit,” so it looks like this isn’t a course-related topic.

If you meant something else related to the course, feel free to rephrase it. Are there other areas you want to explore?


'I couldn’t find any course FAQ entry for “queen gambit,” so it looks like this isn’t a course-related topic.\n\nIf you meant something else related to the course, feel free to rephrase it. Are there other areas you want to explore?'

This is a lightweight form of an input guardrail. We tell the agent what's in scope and what isn't. A real guardrail checks the input before the agent runs and can block off-topic questions outright. That's a separate topic, but instructions are the first place to start.

Every agent framework wraps this same pattern, whether it's LangChain, PydanticAI, or the OpenAI Agents SDK.

## ToyAIKit


The handwritten agent loop from the previous lesson is educational but repetitive. Every time you build a new agent, you'd write the same while-loop, the same function-call handling, the same message management.

ToyAIKit wraps this pattern so you can focus on tools, prompts, and behavior. We built it together in a DataTalks.Club workshop a while back. It does the same thing as our handwritten loop with less boilerplate. If you open its runners code, you'll find the same while True loop we wrote by hand.

In [44]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [45]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

### Letting ToyAIKit generate the schema

Writing that schema by hand is annoying, and we don't want to do it for every function. So we don't have to.
If we add a type hint and a docstring to search, ToyAIKit reads them and derives the schema for us:

In [46]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [47]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [48]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

The output is the same JSON schema we hand-wrote in the function calling lesson. ToyAIKit generated it from the docstring and the type hint.

Every modern agent framework does this same trick. It reads a typed Python function with a docstring and builds the schema from it. The OpenAI Agents SDK, PydanticAI, LangChain and Google ADK all work this way. You write the tool and the framework figures out how to describe it.

### Chat interface and runner

In [49]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

The chat_interface handles display in the notebook. The callback renders model messages and tool calls as they happen. The runner runs the agent loop, the same while True we wrote by hand. It sends messages, executes function calls, adds tool outputs back, and repeats until the model is done.

In [50]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [52]:
result.cost

CostInfo(input_cost=Decimal('0.0027675'), output_cost=Decimal('0.0015345'), total_cost=Decimal('0.0043020'))

In [53]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local installation Ollama"}', call_id='call_

In [54]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


The runner picks up where the last call left off, with the same agent loop and an extended history. The model knows "different model" refers to Ollama because it sees the previous turn in memory. Without that history, it would have no idea what we're asking about.

### Interactive chat

In [55]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='How do I run docker in docker?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"docker in docker run docker in docker